# Visualisasi Output Transfer Learning

Notebook ini membaca hasil dari `outputs/transfer_learning/` dan menampilkan ringkasan dalam bentuk tabel dan grafik.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / 'transfer_learning').exists() and (path / 'model.pth.tar').exists():
            return path
    return start

REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'transfer_learning'
assert OUTPUT_DIR.exists(), f'Output directory tidak ditemukan: {OUTPUT_DIR.resolve()}'
print('Repo root:', REPO_ROOT)
print('Output dir:', OUTPUT_DIR)

plt.style.use('default')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 120)

AssertionError: Output directory tidak ditemukan: C:\Users\User\Documents\penelitian xray\CheXNet-Inference\transfer_learning\outputs\transfer_learning

## File Output yang Tersedia

In [ ]:
files = []
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        files.append({
            'file': str(path.relative_to(OUTPUT_DIR)),
            'size_kb': round(path.stat().st_size / 1024, 2),
        })
pd.DataFrame(files)

## Ringkasan Metrik Test dan Validation

In [ ]:
def load_json(name):
    path = OUTPUT_DIR / name
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

test_metrics = load_json('test_metrics.json')
validation_metrics = load_json('validation_metrics.json')

summary_keys = [
    'exact_match_accuracy',
    'micro_precision',
    'micro_recall',
    'micro_f1',
    'macro_precision',
    'macro_recall',
    'macro_f1',
]

rows = []
for key in summary_keys:
    rows.append({
        'metric': key,
        'validation': None if validation_metrics is None else validation_metrics.get(key),
        'test': None if test_metrics is None else test_metrics.get(key),
    })

summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
if not summary_df.empty:
    ax = summary_df.set_index('metric')[['validation', 'test']].plot(kind='bar', figsize=(10, 5))
    ax.set_title('Ringkasan Metrik Validation vs Test')
    ax.set_ylabel('score')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## Metrik Per Kelas

In [ ]:
def per_class_df(metrics, split_name):
    if not metrics or 'per_class' not in metrics:
        return pd.DataFrame()
    rows = []
    for class_name, values in metrics['per_class'].items():
        row = {'split': split_name, 'class_name': class_name}
        row.update(values)
        rows.append(row)
    return pd.DataFrame(rows)

per_class = pd.concat(
    [per_class_df(validation_metrics, 'validation'), per_class_df(test_metrics, 'test')],
    ignore_index=True,
)
per_class

In [ ]:
if not per_class.empty:
    test_per_class = per_class[per_class['split'] == 'test'].set_index('class_name')
    ax = test_per_class[['precision', 'recall', 'f1']].plot(kind='bar', figsize=(11, 5))
    ax.set_title('Metrik Test Per Kelas')
    ax.set_ylabel('score')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## Threshold Hasil Validation Tuning

In [ ]:
thresholds = load_json('thresholds.json') or {}
threshold_df = pd.DataFrame([
    {
        'class_name': class_name,
        'threshold': values.get('threshold') if isinstance(values, dict) else values,
        'validation_f1': values.get('validation_f1') if isinstance(values, dict) else None,
    }
    for class_name, values in thresholds.items()
])
threshold_df

In [ ]:
if not threshold_df.empty:
    ax = threshold_df.set_index('class_name')[['threshold', 'validation_f1']].plot(kind='bar', figsize=(11, 5))
    ax.set_title('Threshold dan F1 Validation Per Kelas')
    ax.set_ylabel('value')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## Distribusi Kelas Training

In [ ]:
class_distribution_path = OUTPUT_DIR / 'class_distribution.csv'
class_distribution = pd.read_csv(class_distribution_path) if class_distribution_path.exists() else pd.DataFrame()
class_distribution

In [ ]:
if not class_distribution.empty:
    ax = class_distribution.set_index('class_name')[['positive_count', 'negative_count']].plot(kind='bar', figsize=(11, 5))
    ax.set_title('Distribusi Label Training')
    ax.set_ylabel('count')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

    ax = class_distribution.set_index('class_name')['pos_weight'].plot(kind='bar', figsize=(11, 4), color='tab:orange')
    ax.set_title('pos_weight Untuk BCEWithLogitsLoss')
    ax.set_ylabel('pos_weight')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## Training Log Stage A dan Stage B

In [ ]:
training_log_path = OUTPUT_DIR / 'training_log.csv'
training_log = pd.read_csv(training_log_path) if training_log_path.exists() else pd.DataFrame()
training_log

In [ ]:
if not training_log.empty:
    plot_df = training_log.copy()
    plot_df['epoch_label'] = plot_df['stage'].astype(str) + plot_df['epoch'].astype(str)
    ax = plot_df.plot(x='epoch_label', y=['train_loss', 'val_loss'], marker='o', figsize=(12, 5))
    ax.set_title('Training dan Validation Loss')
    ax.set_xlabel('stage/epoch')
    ax.set_ylabel('loss')
    ax.grid(alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    best_rows = training_log.loc[training_log.groupby('stage')['val_loss'].idxmin()].reset_index(drop=True)
    display(best_rows)

## Prediksi Test

In [ ]:
predictions_path = OUTPUT_DIR / 'test_predictions.csv'
predictions = pd.read_csv(predictions_path) if predictions_path.exists() else pd.DataFrame()
predictions.head(20)

In [ ]:
if not predictions.empty:
    pred_label_counts = predictions['predicted_labels'].fillna('').str.split('|').explode()
    pred_label_counts = pred_label_counts[pred_label_counts != ''].value_counts().rename_axis('class_name').reset_index(name='predicted_count')

    true_label_counts = predictions['ground_truth_labels'].fillna('').str.split('|').explode()
    true_label_counts = true_label_counts[true_label_counts != ''].value_counts().rename_axis('class_name').reset_index(name='true_count')

    label_counts = pd.merge(true_label_counts, pred_label_counts, on='class_name', how='outer').fillna(0)
    label_counts[['true_count', 'predicted_count']] = label_counts[['true_count', 'predicted_count']].astype(int)
    display(label_counts)

    ax = label_counts.set_index('class_name')[['true_count', 'predicted_count']].plot(kind='bar', figsize=(10, 5))
    ax.set_title('Jumlah Label Ground Truth vs Prediksi Pada Test Set')
    ax.set_ylabel('count')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## Grad-CAM Output

In [ ]:
gradcam_dir = OUTPUT_DIR / 'gradcam'
gradcam_index_path = gradcam_dir / 'gradcam_index.csv'
gradcam_index = pd.read_csv(gradcam_index_path) if gradcam_index_path.exists() else pd.DataFrame()
gradcam_index.head(20)

In [ ]:
from PIL import Image

image_files = sorted([p for p in gradcam_dir.glob('*.jpg')]) if gradcam_dir.exists() else []
image_files = image_files[:6]

if image_files:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for ax, image_path in zip(axes, image_files):
        ax.imshow(Image.open(image_path))
        ax.set_title(image_path.name, fontsize=8)
        ax.axis('off')
    for ax in axes[len(image_files):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada file Grad-CAM .jpg di', gradcam_dir)